# Entrenamiento del modelo

### Importación de librerías

In [ ]:
import sys
sys.path.append("..")
from scripts.carga_modelo import cargar_modelo
import pandas as pd
import torch
from sklearn.model_selection import GroupShuffleSplit
from datasets import Dataset
from transformers import DataCollatorForSeq2Seq, TrainingArguments, Trainer
import time
import gc

### Rutas

In [ ]:
datos_limpios = '../datos/entrenamiento.csv'
esquemas = '../esquemas/'

primer_modelo = '../modelos/modelo_r8'
resultados_primer_modelo = '../resultados/salida_r8'
historial_primer_modelo = '../resultados/historial_r8.csv'

segundo_modelo = '../modelos/modelo_r16'
resultados_segundo_modelo = '../resultados/salida_r16'
historial_segundo_modelo = '../resultados/historial_r16.csv'

### Importación de datos

In [3]:
datos = pd.read_csv(datos_limpios)
datos = datos.drop(columns=['valida'])
datos.head(2)

,db_id,pregunta,sql
0,department_management,¿Cuántos jefes de departamento tienen más de 5...,SELECT count(*) FROM head WHERE age > 56
1,department_management,"Indique el nombre, estado de nacimiento y edad...","SELECT name, born_state, age FROM head ORDER..."


### Agregando el esquema a los datos

In [5]:
# Función para leer el esquema.
def obtener_esquema(id_esquema):
    ruta = esquemas +  f'{id_esquema}.sql'

    with open(ruta, 'r', encoding='utf-8') as archivo:
        return archivo.read()

datos['esquema'] = datos['db_id'].apply(obtener_esquema)
datos.head(2)

,db_id,pregunta,sql,esquema
0,department_management,¿Cuántos jefes de departamento tienen más de 5...,SELECT count(*) FROM head WHERE age > 56,CREATE TABLE `department` (`Department_ID` DEC...
1,department_management,"Indique el nombre, estado de nacimiento y edad...","SELECT name, born_state, age FROM head ORDER...",CREATE TABLE `department` (`Department_ID` DEC...


### División del conjunto de entrenamiento y prueba

In [6]:
SEMILLA = 2026
division = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=SEMILLA)
idx_train, idx_test = next(division.split(datos, groups=datos['db_id']))

# Entrenamiento
df_train = datos.iloc[idx_train].reset_index(drop=True)
train_dataset = Dataset.from_pandas(df_train)

# Validación
df_test = datos.iloc[idx_test].reset_index(drop=True).copy()
test_dataset = Dataset.from_pandas(df_test)

### Configuraciones de LoRA

Se realizará un cambio en el parámetro `r` (8 y 16).  

**Hipótesis**  
Aumentar el parámetro `r` incrementará la capacidad de adaptación del modelo y mejorará la generación de consultas SQL. La desventaja es que aumentará la cantidad de parámetros entrenables.

### Función que Tokeniza las entradas y salidas

In [7]:
MAX_TOKENS = 1500

def tokenizar_ejemplo(tokenizer, fila):

    mensajes = [{'role': 'system',
                 'content': ('Genera una única consulta SQL compatible con MySQL 8.4.10.' 
                             'Devuelve solamente la consulta SELECT o WITH ... SELECT.')},
                {'role': 'user',
                 'content': (f'Esquema:\n{fila["esquema"]}\n\n'
                             f'Pregunta:\n{fila["pregunta"]}')}]

    # Tokens de entrada
    entrada = tokenizer.apply_chat_template(mensajes, tokenize=True, add_generation_prompt=True)

    # Tokens de la respuesta correcta
    salida = tokenizer(fila['sql'].strip(), add_special_tokens=False)['input_ids'] + [tokenizer.eos_token_id]

    return {'input_ids': entrada + salida,
            'attention_mask': [1] * (len(entrada) + len(salida)),
            'labels': [-100] * len(entrada) + salida}

## Primer modelo

### Carga del modelo

In [8]:
modelo, tokenizer = cargar_modelo(r=8)
modelo.print_trainable_parameters()

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


### Tokenización de datos

In [9]:
train_tokens = train_dataset.map(lambda fila: tokenizar_ejemplo(tokenizer, fila), remove_columns=train_dataset.column_names)
test_tokens = test_dataset.map(lambda fila: tokenizar_ejemplo(tokenizer, fila), remove_columns=test_dataset.column_names)

longitudes = [len(x) for x in train_tokens['input_ids']]
print(f'Mayores a {MAX_TOKENS}:', sum(x > MAX_TOKENS for x in longitudes))

# Filtrando los que tienen demasiados tokens.
train_tokens = train_tokens.filter(lambda fila: len(fila["input_ids"]) <= MAX_TOKENS)
test_tokens = test_tokens.filter(lambda fila: len(fila["input_ids"]) <= MAX_TOKENS)

Map:   0%|          | 0/4268 [00:00<?, ? examples/s]

Map:   0%|          | 0/475 [00:00<?, ? examples/s]

Mayores a 1500: 64


Filter:   0%|          | 0/4268 [00:00<?, ? examples/s]

Filter:   0%|          | 0/475 [00:00<?, ? examples/s]

Se filtraron los mayores a 1,500 tokens debido a su baja cantidad y que pueden suponer tiempos mucho más largos para predicción.

### Data Collator

In [10]:
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True,
                                  label_pad_token_id=-100, pad_to_multiple_of=8)

### Parámetros del entrenamiento

In [11]:
# Mejor eficiencia del CPU
modelo.config.use_cache = False

argumentos = TrainingArguments(output_dir=resultados_primer_modelo,
                               num_train_epochs=2,
                               per_device_train_batch_size=1,
                               per_device_eval_batch_size=1,
                               gradient_accumulation_steps=8,
                               learning_rate=2e-4,
                               weight_decay=0.01,
                               eval_strategy='epoch',
                               save_strategy='epoch',
                               logging_steps=20,
                               fp16=True,
                               gradient_checkpointing=True,
                               save_total_limit=2,
                               report_to='none',
                               seed=SEMILLA)

### Entrenador del modelo

In [12]:
trainer = Trainer(model=modelo,
                  args=argumentos,
                  train_dataset=train_tokens,
                  eval_dataset=test_tokens,
                  data_collator=collator)

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


### Entrenamiento del modelo

In [13]:
inicio = time.perf_counter()

trainer.train()

tiempo = time.perf_counter() - inicio

print(f'Entrenamiento finalizado en {tiempo / 60:.2f} minutos')

Epoch,Training Loss,Validation Loss
1,0.049000,0.186114


Entrenamiento finalizado en 94.16 minutos


### Evaluando el error

In [14]:
resultados = trainer.evaluate(eval_dataset=test_tokens)
print('Validation Loss:', resultados['eval_loss'])

Validation Loss: 0.18611358106136322


### Guardando el modelo

In [15]:
modelo.save_pretrained(primer_modelo)
tokenizer.save_pretrained(primer_modelo)

('../modelos/modelo_r8/tokenizer_config.json',
 '../modelos/modelo_r8/special_tokens_map.json',
 '../modelos/modelo_r8/vocab.json',
 '../modelos/modelo_r8/merges.txt',
 '../modelos/modelo_r8/added_tokens.json',
 '../modelos/modelo_r8/tokenizer.json')

### Guardando el historial del modelo

In [16]:
historial = pd.DataFrame(trainer.state.log_history)
historial.to_csv(historial_primer_modelo, index=False)

### Liberando memoria

In [17]:
del trainer
del modelo
del tokenizer
del collator
del argumentos
del historial
del resultados

# Liberar memoria RAM
gc.collect()

# Liberar memoria caché de la GPU
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Segundo modelo

### Carga del modelo

In [18]:
modelo, tokenizer = cargar_modelo(r=16)
modelo.print_trainable_parameters()

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


### Tokenización de datos

In [19]:
train_tokens = train_dataset.map(lambda fila: tokenizar_ejemplo(tokenizer, fila), remove_columns=train_dataset.column_names)
test_tokens = test_dataset.map(lambda fila: tokenizar_ejemplo(tokenizer, fila), remove_columns=test_dataset.column_names)

longitudes = [len(x) for x in train_tokens['input_ids']]
print(f'Mayores a {MAX_TOKENS}:', sum(x > MAX_TOKENS for x in longitudes))

# Filtrando los que tienen demasiados tokens.
train_tokens = train_tokens.filter(lambda fila: len(fila["input_ids"]) <= MAX_TOKENS)
test_tokens = test_tokens.filter(lambda fila: len(fila["input_ids"]) <= MAX_TOKENS)

Map:   0%|          | 0/4268 [00:00<?, ? examples/s]

Map:   0%|          | 0/475 [00:00<?, ? examples/s]

Mayores a 1500: 64


Filter:   0%|          | 0/4268 [00:00<?, ? examples/s]

Filter:   0%|          | 0/475 [00:00<?, ? examples/s]

Se filtraron los mayores a 1,500 tokens debido a su baja cantidad y que pueden suponer tiempos mucho más largos para predicción.

### Data Collator

In [20]:
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True,
                                  label_pad_token_id=-100, pad_to_multiple_of=8)

### Parámetros del entrenamiento

In [21]:
# Mejor eficiencia del CPU
modelo.config.use_cache = False

argumentos = TrainingArguments(output_dir=resultados_segundo_modelo,
                               num_train_epochs=2,
                               per_device_train_batch_size=1,
                               per_device_eval_batch_size=1,
                               gradient_accumulation_steps=8,
                               learning_rate=2e-4,
                               weight_decay=0.01,
                               eval_strategy='epoch',
                               save_strategy='epoch',
                               logging_steps=20,
                               fp16=True,
                               gradient_checkpointing=True,
                               save_total_limit=2,
                               report_to='none',
                               seed=SEMILLA)

### Entrenador del modelo

In [22]:
trainer = Trainer(model=modelo,
                  args=argumentos,
                  train_dataset=train_tokens,
                  eval_dataset=test_tokens,
                  data_collator=collator)

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


### Entrenamiento del modelo

In [23]:
inicio = time.perf_counter()

trainer.train()

tiempo = time.perf_counter() - inicio

print(f'Entrenamiento finalizado en {tiempo / 60:.2f} minutos')

Epoch,Training Loss,Validation Loss
1,0.048500,0.182054


Entrenamiento finalizado en 91.23 minutos


### Evaluando el error

In [24]:
resultados = trainer.evaluate(eval_dataset=test_tokens)
print('Validation Loss:', resultados['eval_loss'])

Validation Loss: 0.1820542812347412


### Guardando el modelo

In [25]:
modelo.save_pretrained(segundo_modelo)
tokenizer.save_pretrained(segundo_modelo)

('../modelos/modelo_r16/tokenizer_config.json',
 '../modelos/modelo_r16/special_tokens_map.json',
 '../modelos/modelo_r16/vocab.json',
 '../modelos/modelo_r16/merges.txt',
 '../modelos/modelo_r16/added_tokens.json',
 '../modelos/modelo_r16/tokenizer.json')

### Guardando el historial del modelo

In [26]:
historial = pd.DataFrame(trainer.state.log_history)
historial.to_csv(historial_segundo_modelo, index=False)

### Liberando memoria

In [27]:
del trainer
del modelo
del tokenizer
del collator
del argumentos
del historial
del resultados

# Liberar memoria RAM
gc.collect()

# Liberar memoria caché de la GPU
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Conclusiones

El segundo modelo tuvo mejor rendimiento en el conjunto de validación (no propio) por 0.00406 respecto a la función de pérdida, pero este modelo tiene el doble de parámetros entrenables. Por estas razones, la hipótesis se cumplió. 
No se decidirá aún por el mejor modelo, ya que se evaluarán los tiempos y errores en cada uno de los modelos (en otro cuaderno) y ahí se tomará la decisión.